# Watching over a power transformer: what generated time series are for

A grid operator runs power transformers. Each one logs its **load** and its **oil temperature** (`OT`) every hour, and the operator relies on two models:

- a **forecaster** that predicts the oil temperature of the next 24 hours, to anticipate overheating;
- an **anomaly detector** that must raise an alert when a transformer misbehaves (cooling degradation, cooling cycling), and stay silent otherwise.

The recorded data is real but narrow: two transformers, two years, **no failure at all**. The transformations of `kcai-data-sampling-timeseries` create the situations nobody recorded, and this notebook uses them to answer two questions:

1. **Forecasting**: how much worse does the forecast get when the history is degraded (noisy probe, coarser logging)?
2. **Anomaly detection**: which faults does the detector catch, from what size, and does it stay quiet on harmless variability?

**Data.** [ETT](https://github.com/zhouhaoyi/ETDataset) (*Electricity Transformer Temperature*, Zhou et al., Informer, AAAI 2021), the reference benchmark of forecasting: `ETTh1` and `ETTh2`, hourly, July 2016 to June 2018. ETT is licensed **CC BY-ND 4.0**: the data is downloaded by `scripts/fetch_ett_sample.py`, and the outputs of this notebook (generated windows, figures) are adapted material, **not to be committed or shared** outside the team.

**Models.** Deliberately simple stand-ins for the operator's models, in `ett_models.py` next to this notebook. They need scikit-learn, which is not a workspace dependency:

```bash
uv run --with scikit-learn --with matplotlib --with ipykernel jupyter lab
```

## Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

%config InlineBackend.figure_format = 'png'

# Work from the repository root: every path in the configs is relative to it.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO / "examples" / "notebooks"))

import ett_models as em
from kcai_data_sampling_job.cli import run

DATA = Path("examples/data/ett_sample/series.parquet")
if not DATA.exists():
    subprocess.run([sys.executable, "scripts/fetch_ett_sample.py"], check=True)

# Figures are also saved as PNG for the slides (git-ignored, ETT-derived: not to be shared).
FIGURES = Path("examples/outputs/ett-demo/figures")
FIGURES.mkdir(parents=True, exist_ok=True)



def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=160, bbox_inches="tight")

Both example configs read `OT` alone (channel 6), cut it into **one-week windows (168 hours) every day**, and apply their transformations to every window. Each run writes a ledger (one row per generated window: source window, algorithm, parameters, seed) and one `.npy` file per window.

In [ ]:
for name in ("ett-anomaly", "ett-forecasting"):
    with open(f"examples/config/{name}.yaml") as f:
        config = yaml.safe_load(f)
    shutil.rmtree(f"examples/outputs/{name}", ignore_errors=True)
    print(name, run(config))

## 1. The data

Two years of hourly oil temperature, one series per transformer. Everything the models learn comes from the first 70 % of each series; every result below is measured on windows that start after it, so no evaluated window overlaps the training data.

In [ ]:
SERIES = em.load_ot(DATA)
ENDS = em.train_end(SERIES)
START = pd.Timestamp("2016-07-01")

fig, axes = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True, sharey=True)
for ax, (sid, ot) in zip(axes, SERIES.items()):
    t = pd.date_range(START, periods=len(ot), freq="h")
    ax.plot(t, ot, lw=0.4)
    ax.axvspan(t[ENDS[sid]], t[-1], color="gray", alpha=0.06, lw=0)
    ax.axvline(t[ENDS[sid]], color="gray", lw=1)
    ax.text(t[ENDS[sid]], 0.97, "  evaluation", transform=ax.get_xaxis_transform(), fontsize=8, va="top")
    ax.set_title(f"{sid}: oil temperature", loc="left")
    ax.set_ylabel("°C")
fig.tight_layout()
save(fig, "1_data")
plt.show()

## 2. Transformations as situations

One evaluation week of `ETTh2` (spring 2018), as recorded (blue) and as generated (orange). Each transformation stands for something that happens to a real transformer:

| Transformation | Situation | Role in the detection test |
|---|---|---|
| `jitter`, `sigma` 0.3 °C | noisy temperature probe | normal: must **not** raise an alert |
| `low_pass`, `cutoff` 0.1 cycle/h | readings averaged by the logging system | normal |
| `drift`, `slope` in °C per hour | cooling degradation: the oil warms up for the same load | fault: **must** raise an alert |
| `add_sinusoid`, 1 cycle per 6 h | cooling system cycling on and off | fault |

In [ ]:
ANOMALY, ANOMALY_WINDOWS = em.load_outputs("examples/outputs/ett-anomaly/ett_ot", SERIES)

# A spring week of ETTh2 with a clear daily cycle, and none of the runs of exact zeros ETT has in places.
SID = "ETTh2"
SPRING = int((pd.Timestamp("2018-04-02") - START) / pd.Timedelta(hours=1))
WEEK = next(
    s for s in sorted(ANOMALY.start_window[ANOMALY.series_id == SID].unique())
    if s >= SPRING and (SERIES[SID][s : s + 168] != 0).all()
)
SHOWN = [
    ("jitter", 0.3, "Noisy probe (jitter, σ = 0.3 °C)"),
    ("low_pass", 0.1, "Coarser logging (low_pass, cutoff 0.1 cycle/h)"),
    ("drift", 0.11, "Cooling degradation (drift, +0.11 °C/h)"),
    ("add_sinusoid", 10.0, "Cooling cycling (6 h cycle, amplitude 10 °C)"),
]
in_week = (ANOMALY.series_id == SID) & (ANOMALY.start_window == WEEK)
original = SERIES[SID][WEEK : WEEK + 168]
hours = np.arange(168)

fig, axes = plt.subplots(2, 2, figsize=(10, 5.2), sharex=True, sharey=True)
for ax, (algorithm, value, title) in zip(axes.flat, SHOWN):
    row = ANOMALY.index[in_week & (ANOMALY.algorithm == algorithm) & np.isclose(ANOMALY.value, value)][0]
    ax.plot(hours, original, lw=1.2, label="recorded")
    ax.plot(hours, ANOMALY_WINDOWS[row], lw=1.2, label="generated")
    ax.set_title(title, loc="left", fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("hour of the week")
    ax.set_xticks(range(0, 169, 24))
for ax in axes[:, 0]:
    ax.set_ylabel("°C")
axes[0, 0].legend(loc="upper left")
fig.suptitle(f"{SID}, week starting {START + pd.Timedelta(hours=int(WEEK)):%Y-%m-%d}", x=0.01, ha="left", fontsize=9)
fig.tight_layout()
save(fig, "2_week")
plt.show()

# Every generated window is traceable: its ledger row says where it comes from and what was done to it.
ANOMALY.loc[in_week, ["series_id", "start_window", "end_window", "algorithm", "params", "seed", "artifact"]]

## 3. Forecasting: how much does a degraded history cost?

Two forecasters predict the next 24 hours from a one-week history:

- **seasonal naive**: tomorrow is today;
- **lag ridge**: a ridge regression on the week, fitted on clean training windows.

Each generated window is a degraded history; the **true future is read from the original series** after the window's end, so it is never transformed. We measure the mean absolute error of the 24-hour forecast as the degradation grows.

In [ ]:
HISTORIES, FUTURES = em.training_set(SERIES, 168)
MODELS = {
    "lag ridge": em.LagRidge().fit(HISTORIES, FUTURES),
    "seasonal naive": em.SeasonalNaive().fit(HISTORIES, FUTURES),
}
FORECAST, FORECAST_WINDOWS = em.load_outputs("examples/outputs/ett-forecasting/ett_ot", SERIES)
ERRORS = em.forecast_errors(FORECAST, FORECAST_WINDOWS, SERIES, MODELS)
TABLE = ERRORS.groupby(["algorithm", "value", "model"]).mae.mean().unstack().round(2)
TABLE

In [ ]:
PANELS = [
    ("jitter", "Noisy probe", "noise σ (°C)", False),
    ("low_pass", "Coarser logging", "cutoff (cycles per hour), coarser to the right", True),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (algorithm, title, xlabel, invert) in zip(axes, PANELS):
    curve = TABLE.loc[algorithm]
    for model in MODELS:
        ax.plot(curve.index, curve[model], marker="o", ms=5, label=model)
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    if invert:
        ax.invert_xaxis()
axes[0].set_ylabel("forecast error, next 24 h (°C)")
axes[0].set_ylim(bottom=0)
axes[0].legend(loc="lower right")
fig.tight_layout()
save(fig, "3_forecasting")
plt.show()

The error grows steadily with the probe noise, for both models. The ridge, the better model on clean data, is also the one that loses most when the logging gets coarser: it relies on the hour-to-hour detail that a smoothed history no longer has.

## 4. Anomaly detection: what does the detector see?

The detector predicts the last 24 hours of each week from the 144 hours before them; the anomaly score is how badly it does. It raises an alert above the 95th percentile of the scores on clean training weeks, so about 5 % of normal weeks raise an alert by construction.

ETT has **no labelled anomaly**. The label comes from the ledger instead: we know which fault was injected, of what size, into which window.

In [ ]:
DETECTOR = em.SurpriseDetector(window=168).fit(SERIES)
ANOMALY["alert"] = DETECTOR.alerts(ANOMALY_WINDOWS)
CLEAN = em.clean_windows(ANOMALY.drop_duplicates(["series_id", "start_window"]), SERIES)
FALSE_ALARMS = DETECTOR.alerts(CLEAN).mean()
RATES = ANOMALY.groupby(["algorithm", "value"]).alert.mean()

print(f"alert threshold: {DETECTOR.threshold:.2f} °C mean error")
pd.DataFrame({
    "windows": ["recorded weeks", "noisy probe (jitter 0.3)", "coarser logging (low_pass 0.1)"],
    "alert rate": [FALSE_ALARMS, RATES.loc[("jitter", 0.3)], RATES.loc[("low_pass", 0.1)]],
}).style.format({"alert rate": "{:.1%}"}).hide()

Harmless variability does not raise more alerts than the recorded weeks: the detector is quiet where it should be. Now the faults, from small to large:

In [ ]:
NATURAL_SLOPE = 0.07  # 95th percentile of weekly OT slopes on ETTh2, °C per hour
FAULTS = [
    ("drift", "Cooling degradation", "drift (°C per hour)"),
    ("add_sinusoid", "Cooling cycling (6 h cycle)", "amplitude (°C)"),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (algorithm, title, xlabel) in zip(axes, FAULTS):
    curve = RATES.loc[algorithm] * 100
    ax.axhline(FALSE_ALARMS * 100, color="gray", lw=1, ls="--")
    ax.plot(curve.index, curve.values, marker="o", ms=5)
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    ax.set_ylim(0, 105)
axes[0].axvspan(0, NATURAL_SLOPE, color="gray", alpha=0.08, lw=0)
axes[0].text(NATURAL_SLOPE / 2, 95, "natural\nweekly slopes", ha="center", va="top", fontsize=8)
axes[1].text(axes[1].get_xlim()[1], FALSE_ALARMS * 100 + 2, "false alarms on recorded weeks ", ha="right", color="gray", fontsize=8)
axes[0].set_ylabel("weeks raising an alert (%)")
fig.tight_layout()
save(fig, "4_anomaly")
plt.show()

- **Cooling degradation** is caught once it clearly exceeds what the weather and the load already do to the oil temperature: a drift inside the natural range is invisible, a strong one is caught most of the time.
- **Cooling cycling** is nearly invisible until it is very large. The detector sees the oscillation in the history and *extrapolates* it, instead of being surprised by it: a regular fault is a predictable one. This is a blind spot of every forecast-based detector, and the generated faults reveal it **before a real failure does**.

## 5. What this shows

- **The same transformations serve both use cases.** For forecasting they degrade the input to measure robustness; for detection they play either harmless variability or a fault. The scenario gives the meaning, not the transformation.
- **Every generated window is traceable**: the ledger records its source window, the algorithm, its parameters and seed. That is what turns generated data into labelled test data.
- **Sweeping a parameter gives a curve, not a score**: how fast the forecast degrades, from what size a fault is caught, and where the detector is blind.